# Agentic Review with Skills, Memory, and Helper Agents

This comprehensive tutorial covers the full agentic capabilities of LatteReview v2:

1. **Search Skills** — enabling reviewers to search DuckDuckGo, PubMed, and other sources
2. **Memory Skills** — persistent cross-item learning within a review batch
3. **Helper Agents** — multi-agent collaboration where a lead reviewer consults domain experts
4. **Custom Skills** — building and registering your own tool skills

**Requirements:** `OPENAI_API_KEY` environment variable must be set.

## Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import json
import os
from pathlib import Path
from lattereview.agentic import AgenticReviewer, ScoringReviewer

## Built-in Skills Overview

LatteReview v2 provides 9 built-in skills that extend reviewer capabilities:

| Skill | Type | Description |
|-------|------|-------------|
| `searching-duckduckgo` | Search | Search the web via DuckDuckGo (no API key needed) |
| `searching-pubmed` | Search | Search PubMed for biomedical literature |
| `searching-semantic-scholar` | Search | Search Semantic Scholar for academic papers |
| `searching-arxiv` | Search | Search arXiv for preprints |
| `searching-google` | Search | Search via Google (requires API key) |
| `searching-content` | Search | Search within the provided content/text |
| `managing-memory` | State | Save and retrieve notes across items in a batch |
| `flagging-items` | State | Flag items for human review when uncertain |
| `discussing-with-helpers` | Multi-agent | Consult helper agents for specialized opinions |

Skills are enabled by passing their names to the `skills` parameter. The reviewer's agentic loop then has access to these tools alongside its reasoning.

## Load the Dataset

In [2]:
df = pd.read_csv("data.csv")
print(f"Dataset: {len(df)} articles on agent-based modeling")
print(f"Columns: {list(df.columns)}")
df[["Title", "Year"]].head(5)

Dataset: 20 articles on agent-based modeling
Columns: ['Title', 'Abstract', 'Authors', 'Year']


,Title,Year
0,Fusing an agent-based model of mosquito popula...,2022
1,PDRL: Multi-Agent based Reinforcement Learning...,2023
2,Learning-accelerated Discovery of Immune-Tumou...,2019
3,Investigating spatiotemporal dynamics and sync...,2018
4,Modeling the Spread of COVID-19 in University ...,2024


---

## Section 1: Search Skills

Search skills allow the reviewer to look up additional context during its agentic loop. This is particularly useful when:
- The abstract makes claims that need verification
- The reviewer needs domain context to assess quality
- Citations or related work would inform the score

We create a `ScoringReviewer` with `searching-duckduckgo` and `searching-pubmed` skills, `max_iterations=15`, and `agentic_effort="high"` to give the agent plenty of room to research.

In [3]:
search_reviewer = ScoringReviewer(
    name="ResearchAnalyst",
    backstory=(
        "You are a computational biology researcher who evaluates agent-based "
        "modeling studies. When reviewing a paper, you actively search for related "
        "work, verify claims, and check whether the described methodology is "
        "well-established. You use DuckDuckGo for general context and PubMed "
        "for finding related biomedical literature."
    ),
    model="openai:gpt-5.4-mini",
    scoring_task=(
        "Rate the methodological rigor and novelty of this agent-based modeling study. "
        "Search for related work to contextualize the contribution."
    ),
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules=(
        "1=poor methodology with no novelty, "
        "2=weak methodology or incremental contribution, "
        "3=adequate methodology with some novelty, "
        "4=strong methodology with clear contribution, "
        "5=excellent methodology with significant novelty"
    ),
    max_iterations=15,
    agentic_effort="high",
    skills=["searching-duckduckgo", "searching-pubmed"],
)

print(f"Reviewer: {search_reviewer.name}")
print(f"Skills: {search_reviewer.skills}")
print(f"Max iterations: {search_reviewer.max_iterations}")
print(f"Agentic effort: {search_reviewer.agentic_effort}")

Reviewer: ResearchAnalyst
Skills: ['searching-duckduckgo', 'searching-pubmed']
Max iterations: 15
Agentic effort: high


In [4]:
# Prepare 3 articles for review
articles_for_search = [
    f"Title: {row['Title']}\n\nAbstract: {row['Abstract']}"
    for _, row in df.head(3).iterrows()
]

print(f"Reviewing {len(articles_for_search)} articles with search skills...\n")

search_results, search_cost = await search_reviewer.review_items(articles_for_search)

print(f"\nTotal cost: ${search_cost:.4f}")
print()

for i, result in enumerate(search_results):
    title = df.iloc[i]["Title"][:70]
    print(f"\n--- Article {i + 1}: {title}... ---")
    print(f"  Score: {result.get('score')}")
    print(f"  Certainty: {result.get('certainty')}")
    reasoning = result.get('reasoning', '')
    # Show first 300 chars of reasoning to see search references
    print(f"  Reasoning: {reasoning[:300]}...")

Reviewing 3 articles with search skills...



/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


...

/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


/home/pouria/projects/lattereview/lattereview/agentic/skills/builtin/searching-duckduckgo/tools.py:35: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Total cost: $0.0000


--- Article 1: Fusing an agent-based model of mosquito population dynamics with a sta... ---
  Score: 4
  Certainty: 87
  Reasoning: The study appears methodologically solid and clearly grounded in existing modeling practice. It combines a mechanistic agent-based model of Ae. aegypti with a statistical GAM-based reconstruction to calibrate one key parameter, which is a sensible hybrid approach for fitting noisy spatio-temporal ab...

--- Article 2: PDRL: Multi-Agent based Reinforcement Learning for Predictive Monitori... ---
  Score: 2
  Certainty: 77
  Reasoning: The described study proposes a multi-agent reinforcement learning framework (multiple DQN agents) layered on top of BiLSTM-based time-series prediction for heart rate, respiration, and temperature. The novelty appears to be mostly in packaging standard components into a monitoring pipeline rather th...

--- Article 3: Learning-accelerated Discovery of Immune-Tumour Interactions... ---
  Score: 4
  Certa

---

## Section 2: Memory Skills

The `managing-memory` skill lets the reviewer save and retrieve notes across items. This enables cross-item learning — for example, the reviewer might notice that several articles use the same framework and save a note about its strengths/weaknesses.

Memories are stored in a `working_dir` and persist across items within a batch. The memory store uses a simple index with title/brief/content for each memory entry.

In [5]:
import shutil

MEMORY_DIR = Path("./memory_demo")
if MEMORY_DIR.exists():
    shutil.rmtree(MEMORY_DIR)

memory_reviewer = ScoringReviewer(
    name="MemoryReviewer",
    backstory=(
        "You are a systematic reviewer evaluating agent-based modeling papers. "
        "As you review each paper, save important observations about methodological "
        "patterns, common frameworks, and recurring themes. Use your accumulated "
        "knowledge from previous papers to inform your assessments of later ones."
    ),
    model="openai:gpt-5.4-mini",
    scoring_task=(
        "Rate the overall quality of this ABM study. Consider methodology, validation, "
        "and contribution. Save any generalizable observations to memory for future reference."
    ),
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules=(
        "1=very poor, 2=below average, 3=average, 4=above average, 5=excellent"
    ),
    max_iterations=10,
    agentic_effort="medium",
    skills=["managing-memory"],
)

# Prepare 5 articles
articles_for_memory = [
    f"Title: {row['Title']}\n\nAbstract: {row['Abstract']}"
    for _, row in df.head(5).iterrows()
]

print(f"Reviewing {len(articles_for_memory)} articles with memory skill...")
memory_results, memory_cost = await memory_reviewer.review_items(
    articles_for_memory,
    working_dir=MEMORY_DIR,
)

print(f"\nTotal cost: ${memory_cost:.4f}")
for i, result in enumerate(memory_results):
    print(f"  Article {i + 1}: score={result.get('score')}, certainty={result.get('certainty')}")

Reviewing 5 articles with memory skill...



Total cost: $0.0000
  Article 1: score=4, certainty=90
  Article 2: score=2, certainty=86
  Article 3: score=4, certainty=84
  Article 4: score=4, certainty=84
  Article 5: score=4, certainty=87


In [6]:
# Inspect the memory store
memory_index_paths = list(MEMORY_DIR.glob("**/memory/_index.json"))

for index_path in memory_index_paths:
    print(f"Memory index: {index_path.relative_to(MEMORY_DIR)}")
    with open(index_path) as f:
        raw = json.load(f)

    # The index is a dict with a "memories" key
    memories = raw.get("memories", raw) if isinstance(raw, dict) else raw

    print(f"Total memories saved: {len(memories)}\n")
    for entry in memories:
        mem_id = entry.get("id", "?")
        brief = entry.get("brief", entry.get("title", ""))
        print(f"  [{mem_id}] {brief}")

        # Read the full memory file (.md format)
        mem_file = index_path.parent / f"{mem_id}.md"
        if mem_file.exists():
            content = mem_file.read_text().strip()
            print(f"    Content: {content[:250]}")
        print()

Memory index: round_A/agent_MemoryReviewer/memory/_index.json
Total memories saved: 5

  [mem_001] Large-scale ABMs are stronger when they combine rich mobility/contact data with scenario exploration, but validation against observed outcomes remains a key limiter.
    Content: Across ABM epidemiology papers, a common pattern is that very large synthetic populations and detailed mobility data improve realism and allow spatial analysis, but the study quality depends heavily on whether the model is calibrated and validated ag

  [mem_002] Strong ABMs can combine mechanistic realism with statistical flexibility by calibrating a small number of parameters to empirical spatio-temporal patterns while preserving interpretable intervention response.
    Content: A recurring strong pattern in ABM papers is hybridization: use a mechanistic agent-based core for causal/intervention analysis, but calibrate key parameters against flexible statistical reconstructions (e.g., GAMs) of observed abundance

---

## Section 3: Helper Agents

Helper agents enable multi-agent collaboration. A lead reviewer can consult one or more helper agents during its agentic loop. This is useful when:
- The review requires domain expertise the lead reviewer lacks
- You want a second opinion during assessment
- Different aspects of an article need different types of expertise

Helpers are separate `AgenticReviewer` instances. The `discussing-with-helpers` skill provides the lead reviewer with a tool to send questions to its helpers and receive responses.

In [7]:
# Create a domain expert helper
epidemiology_expert = AgenticReviewer(
    name="EpidemiologyExpert",
    backstory=(
        "You are a senior epidemiologist with 20 years of experience in "
        "infectious disease modeling. You specialize in evaluating whether "
        "agent-based models of disease transmission are realistic, properly "
        "parameterized, and validated against real-world data. You provide "
        "concise expert opinions."
    ),
    model="openai:gpt-5.4-mini",
    max_iterations=5,
)

# Create the lead reviewer with helper access
lead_reviewer = ScoringReviewer(
    name="LeadReviewer",
    backstory=(
        "You are a systematic review lead evaluating agent-based modeling studies. "
        "For studies involving epidemiology or disease modeling, consult your "
        "epidemiology expert helper to get a domain-specific assessment. "
        "Combine their expert opinion with your own methodological evaluation."
    ),
    model="openai:gpt-5.4-mini",
    scoring_task=(
        "Rate the scientific validity of this agent-based modeling study. "
        "If the study involves disease modeling, consult your epidemiology "
        "expert for a domain assessment."
    ),
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules=(
        "1=invalid methodology, "
        "2=questionable validity, "
        "3=acceptable with caveats, "
        "4=solid and well-validated, "
        "5=exemplary methodology"
    ),
    max_iterations=10,
    agentic_effort="medium",
    skills=["discussing-with-helpers"],
    helpers=[epidemiology_expert],
    helper_max_iterations=5,
)

print(f"Lead reviewer: {lead_reviewer.name}")
print(f"Helpers: {[h.name for h in lead_reviewer.helpers]}")
print(f"Helper max iterations: {lead_reviewer.helper_max_iterations}")

Lead reviewer: LeadReviewer
Helpers: ['EpidemiologyExpert']
Helper max iterations: 5


In [8]:
# Review 2 articles with helper consultation
articles_for_helpers = [
    f"Title: {row['Title']}\n\nAbstract: {row['Abstract']}"
    for _, row in df.head(2).iterrows()
]

print(f"Reviewing {len(articles_for_helpers)} articles with helper agent...\n")

helper_results, helper_cost = await lead_reviewer.review_items(articles_for_helpers)

print(f"\nTotal cost: ${helper_cost:.4f}")
print()

for i, result in enumerate(helper_results):
    title = df.iloc[i]["Title"][:70]
    print(f"\n--- Article {i + 1}: {title}... ---")
    print(f"  Score: {result.get('score')}")
    print(f"  Certainty: {result.get('certainty')}")
    print(f"  Reasoning: {result.get('reasoning', '')[:400]}...")

Reviewing 2 articles with helper agent...




Total cost: $0.0000


--- Article 1: Fusing an agent-based model of mosquito population dynamics with a sta... ---
  Score: 3
  Certainty: 91
  Reasoning: 1. The study uses a clearly mechanistic agent-based model for Ae. aegypti population dynamics, which is appropriate for mosquito abundance and intervention modeling.
2. It is supported by a very large empirical dataset (176,352 household aspirator collections over 1999–2011), which strongly improves plausibility and reduces concern about fitting to sparse data.
3. The hybrid design is methodologic...

--- Article 2: PDRL: Multi-Agent based Reinforcement Learning for Predictive Monitori... ---
  Score: 2
  Certainty: 84
  Reasoning: 1) The paper presents a multi-agent DQN framework layered on top of BiLSTM forecasts for heart rate, respiration, and temperature. This is an applied ML/RL setup, but from the abstract it is not clearly a substantive agent-based model of a complex system.
2) The agent role appears to be monitoring predict

---

## Section 4: Creating a Custom Skill

You can create your own skills by providing a directory with two files:

1. **`SKILL.md`** — A markdown file with YAML frontmatter defining the skill's name and description, plus documentation for the LLM
2. **`tools.py`** — A Python file that creates a `FunctionToolset` with one or more tool functions

The tools receive a `RunContext[ReviewDeps]` as their first argument, giving them access to the current item, memory, and other dependencies.

Let's create a simple custom skill that counts words and sentences in the review text.

In [9]:
# Create the custom skill directory structure
skill_dir = Path("./my_skills/word-counter")
skill_dir.mkdir(parents=True, exist_ok=True)

# Write SKILL.md
skill_md_content = """---
name: word-counter
description: Counts words and sentences in the input text. Use this to get quantitative text metrics before scoring.
---

# Word Counter

Provides text length metrics for the input being reviewed.

## Available Tools

- `count_words(text)` — Count the number of words in the given text
- `count_sentences(text)` — Count the number of sentences in the given text

## When to Use

Use these tools when you need to assess the length or completeness of an abstract or article text as part of your quality evaluation.
"""

(skill_dir / "SKILL.md").write_text(skill_md_content)
print(f"Created {skill_dir / 'SKILL.md'}")

Created my_skills/word-counter/SKILL.md


In [10]:
# Write tools.py
tools_py_content = '''"""Custom word-counter skill tools."""

import re
from pydantic_ai import RunContext
from pydantic_ai.toolsets import FunctionToolset

from lattereview.agentic.deps import ReviewDeps

toolset = FunctionToolset()


@toolset.tool
async def count_words(ctx: RunContext[ReviewDeps], text: str) -> str:
    """Count the number of words in the given text.

    Args:
        ctx: Run context with dependencies.
        text: The text to count words in.
    """
    word_count = len(text.split())
    return f"Word count: {word_count}"


@toolset.tool
async def count_sentences(ctx: RunContext[ReviewDeps], text: str) -> str:
    """Count the number of sentences in the given text.

    Args:
        ctx: Run context with dependencies.
        text: The text to count sentences in.
    """
    sentences = re.split(r"[.!?]+", text)
    # Filter out empty strings
    sentences = [s.strip() for s in sentences if s.strip()]
    return f"Sentence count: {len(sentences)}"
'''

(skill_dir / "tools.py").write_text(tools_py_content)
print(f"Created {skill_dir / 'tools.py'}")

# Also create __init__.py (not required but good practice)
(skill_dir / "__init__.py").write_text("")
print(f"Created {skill_dir / '__init__.py'}")

print(f"\nCustom skill directory:")
for f in sorted(skill_dir.iterdir()):
    print(f"  {f.name}")

Created my_skills/word-counter/tools.py
Created my_skills/word-counter/__init__.py

Custom skill directory:
  SKILL.md
  __init__.py
  tools.py


In [11]:
# Use the custom skill with a reviewer
# NOTE: custom_skill_paths points to the PARENT directory containing skill folders
custom_skill_reviewer = ScoringReviewer(
    name="CustomSkillDemo",
    backstory=(
        "You are a reviewer who evaluates research abstracts. Before scoring, "
        "use the word-counter tool to measure the length of the abstract. "
        "Longer, more detailed abstracts may indicate more thorough research reporting."
    ),
    model="openai:gpt-5.4-mini",
    scoring_task=(
        "Rate the completeness of this research abstract's reporting. "
        "Use the word counter and sentence counter tools to get text metrics, "
        "then assess how well the abstract covers: objective, methods, results, and conclusions."
    ),
    scoring_set=[1, 2, 3, 4, 5],
    scoring_rules=(
        "1=severely incomplete (missing multiple sections), "
        "2=incomplete (missing key information), "
        "3=adequate (covers basics), "
        "4=good (thorough coverage), "
        "5=excellent (comprehensive with quantitative results)"
    ),
    max_iterations=10,
    agentic_effort="medium",
    skills=["word-counter"],
    custom_skill_paths=[Path("./my_skills")],  # Parent dir containing word-counter/
)

# Review 2 articles with the custom skill
custom_articles = [
    f"Title: {row['Title']}\n\nAbstract: {row['Abstract']}"
    for _, row in df.head(2).iterrows()
]

print(f"Reviewing with custom skill 'word-counter'...\n")
custom_results, custom_cost = await custom_skill_reviewer.review_items(custom_articles)

print(f"\nTotal cost: ${custom_cost:.4f}")
for i, result in enumerate(custom_results):
    title = df.iloc[i]["Title"][:70]
    print(f"\n--- Article {i + 1}: {title}... ---")
    print(f"  Score: {result.get('score')}")
    print(f"  Certainty: {result.get('certainty')}")
    print(f"  Reasoning: {result.get('reasoning', '')[:300]}...")

Reviewing with custom skill 'word-counter'...




Total cost: $0.0000

--- Article 1: Fusing an agent-based model of mosquito population dynamics with a sta... ---
  Score: 4
  Certainty: 93
  Reasoning: The abstract is long and detailed at 340 words across 17 sentences, which supports thorough reporting. It clearly states the objective: integrating mechanistic and statistical modeling to better understand Ae. aegypti abundance and control effects. The methods are well described, including the data ...

--- Article 2: PDRL: Multi-Agent based Reinforcement Learning for Predictive Monitori... ---
  Score: 4
  Certainty: 87
  Reasoning: The abstract is fairly long at 238 words across 10 sentences, which suggests substantial reporting. It clearly states the objective: proposing a multi-agent predictive deep reinforcement learning framework for monitoring and forecasting. The methods are described with enough detail to understand the...


## Clean Up

In [12]:
import shutil

# Clean up working directories
for d in [MEMORY_DIR, Path("./my_skills")]:
    if d.exists():
        shutil.rmtree(d)
        print(f"Removed {d}")

# Clean up any agent directories
for d in Path(".").glob("agent_*"):
    if d.is_dir():
        shutil.rmtree(d)
        print(f"Removed {d}")

print("Done.")

Removed memory_demo
Removed my_skills
Done.
